# Render 5 Video Stickman Otomatis di Google Colab (T4 GPU)

Notebook render otomatis berkecepatan tinggi untuk **5 video Stickman** secara berurutan tanpa menu manual:
1. **Orang Indonesia Pertama** (`OrangIndoPertama`):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 307,20 detik (9.216 frame, ~5,12 menit) | 95 scenes
2. **Piramida Mesir Kuno** (`PiramidaStickman`):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 310,23 detik (9.307 frame, ~5,17 menit) | 97 scenes
3. **Misteri Rekayasa Taj Mahal** (`TajMahalStickman`):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 384,68 detik (11.540 frame, ~6,41 menit) | 91 scenes
4. **Evolusi Warna Kulit Manusia** (`KulitStickman`):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 371,96 detik (11.159 frame, ~6,20 menit) | 80 scenes
5. **Eksperimen 11 Hari Tak Tidur Randy Gardner** (`TakTidurStickman`):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 378,12 detik (11.344 frame, ~6,30 menit) | 110 scenes

---
### Petunjuk Penggunaan:
1. Pastikan runtime Colab menggunakan **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Jalankan **Sel 1 (Mount Google Drive)** agar hasil render otomatis tersimpan permanen di Google Drive (`remotion-out/`).
3. Jalankan **Sel 2 (Setup Environment)** untuk instalasi otomatis dependensi Chromium, Node.js 20, dan clone repo.
4. Jalankan **Sel 3 (Render Otomatis 5 Video)** — seluruh 5 video akan dirender berturut-turut dengan progres rapi.
5. Jalankan **Sel 4 (Verifikasi & Download)** untuk memeriksa ukuran file dan mengunduh jika diperlukan.


In [ ]:
# Sel 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
out_dir = '/content/drive/MyDrive/remotion-out'
os.makedirs(out_dir, exist_ok=True)
print('Google Drive terhubung! Output video akan disimpan di:', out_dir)


In [ ]:
%%shell
# Sel 2: Setup Environment, Dependensi Chromium & Clone Repo
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak terdeteksi! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install Dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai 100%! Siap merender 5 video Stickman."


In [ ]:
# Sel 3: Render Otomatis 5 Video Stickman Berurutan
import importlib, os, subprocess, sys, time

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
os.makedirs(out_dir, exist_ok=True)

if '/content/vidio/scripts' not in sys.path:
    sys.path.insert(0, '/content/vidio/scripts')

import colab_render
importlib.reload(colab_render)

start_all = time.time()

tasks = [
    ('OrangIndoPertama', 'orang-indo-pertama-1080p.mp4', '1. Orang Indonesia Pertama (~5,12 Menit)'),
    ('PiramidaStickman', 'piramida-mesir-1080p.mp4', '2. Piramida Mesir (~5,17 Menit)'),
    ('TajMahalStickman', 'tajmahal-1080p.mp4', '3. Misteri Taj Mahal (~6,41 Menit)'),
    ('KulitStickman', 'kulit-1080p.mp4', '4. Evolusi Warna Kulit (~6,20 Menit)'),
    ('TakTidurStickman', 'tak-tidur-1080p.mp4', '5. Eksperimen Tak Tidur 1964 (~6,30 Menit)')
]

for idx, (comp_id, out_name, label) in enumerate(tasks, start=1):
    target_path = os.path.join(out_dir, out_name)
    print(f'\n=====================================================================')
    print(f'=== [{idx}/5] MEMULAI RENDER: {label} ===')
    print(f'=== Komposisi: {comp_id} -> {out_name} ===')
    print(f'=====================================================================')
    t0 = time.time()
    colab_render.run_render(
        composition=comp_id,
        out_path=target_path,
        cwd='/content/vidio',
    )
    t_dur = (time.time() - t0) / 60.0
    print(f'>>> [{idx}/5] Selesai dalam {t_dur:.2f} menit! File: {target_path}')

total_min = (time.time() - start_all) / 60.0
print(f'\n[SELESAI 100%] Seluruh 5 video berhasil dirender dalam total {total_min:.2f} menit!')
print('Semua hasil video tersimpan di:', out_dir)


In [ ]:
# Sel 4: Verifikasi Ukuran File & Opsi Download ke Lokal
import os
from google.colab import files

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
video_files = [
    os.path.join(out_dir, 'orang-indo-pertama-1080p.mp4'),
    os.path.join(out_dir, 'piramida-mesir-1080p.mp4'),
    os.path.join(out_dir, 'tajmahal-1080p.mp4'),
    os.path.join(out_dir, 'kulit-1080p.mp4'),
    os.path.join(out_dir, 'tak-tidur-1080p.mp4')
]

print('Daftar file video hasil render di Drive:')
for v in video_files:
    if os.path.exists(v):
        size_mb = os.path.getsize(v) / (1024 * 1024)
        print(f'  [OK] {os.path.basename(v)}: {size_mb:.2f} MB')
    else:
        print(f'  [--] {os.path.basename(v)}: Belum selesai / tidak ditemukan')

# Uncomment baris di bawah ini jika ingin men-download otomatis ke laptop:
# for v in video_files:
#     if os.path.exists(v):
#         files.download(v)
